In [0]:

spark.conf.set("fs.azure.account.auth.type.<storage-account>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<storage-account>.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage-account>.dfs.core.windows.net", "<application-id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage-account>.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage-account>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory-id>/oauth2/token")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_sales_15 = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferSchema", "true")\
          .load("abfs://bronze@adventureworksdatalakeom.dfs.core.windows.net/AdventureWorks_Sales_2015")

In [0]:
df_sales_15.display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21,SO45080,332,14657,1,1,1
2015-01-01,2001-12-05,SO45079,312,29255,4,1,1
2015-01-01,2001-10-29,SO45082,350,11455,9,1,1
2015-01-01,2001-11-16,SO45081,338,26782,6,1,1
2015-01-02,2001-12-15,SO45083,312,14947,10,1,1
2015-01-02,2001-10-12,SO45084,310,29143,4,1,1
2015-01-02,2001-12-18,SO45086,314,18747,9,1,1
2015-01-02,2001-10-09,SO45085,312,18746,9,1,1
2015-01-03,2001-10-03,SO45093,312,18906,9,1,1
2015-01-03,2001-09-29,SO45090,310,29170,4,1,1


In [0]:
df_sales_15.count()

2630

In [0]:
df_sales_15 = df_sales_15.withColumn(
    "OrderNumber",
    regexp_replace(col("OrderNumber"), "S", "T")
)

In [0]:
df_sales_15.display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21,TO45080,332,14657,1,1,1
2015-01-01,2001-12-05,TO45079,312,29255,4,1,1
2015-01-01,2001-10-29,TO45082,350,11455,9,1,1
2015-01-01,2001-11-16,TO45081,338,26782,6,1,1
2015-01-02,2001-12-15,TO45083,312,14947,10,1,1
2015-01-02,2001-10-12,TO45084,310,29143,4,1,1
2015-01-02,2001-12-18,TO45086,314,18747,9,1,1
2015-01-02,2001-10-09,TO45085,312,18746,9,1,1
2015-01-03,2001-10-03,TO45093,312,18906,9,1,1
2015-01-03,2001-09-29,TO45090,310,29170,4,1,1


# Agg Functions

In [0]:
df_sales_15 = df_sales_15.groupBy("ProductKey").agg(
    sum("OrderQuantity").alias("TotalSold")
)

In [0]:
df_sales_15.display()

ProductKey,TotalSold
322,24
362,78
375,83
368,71
385,33
332,31
350,31
336,28
330,24
346,24


In [0]:
df_sales_15.write.format("parquet")\
            .mode("append")\
            .option("path","abfs://silver@adventureworksdatalakeom.dfs.core.windows.net/AdventureWorks_Sales_2015")\
            .save()